# Loan Classification
This notebook covers only:
1. Data preprocessing
2. Stratified train/test split and CSV export
3. Training all 5 required classification models
4. Saving models in the models folder
5. Computing required evaluation metrics

In [9]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COL = "loan_status"

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
DATA_FILE = ROOT / DATA_DIR / "loan_data.csv"
MODELS_DIR = ROOT / "model"
RESULTS_DIR = ROOT / "results"

for folder in [DATA_DIR, MODELS_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Input file exists: {DATA_FILE.exists()}")

Input file exists: True


In [10]:
df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Missing values per column:")
print(df.isna().sum())
print("\nTarget distribution:")
print(df[TARGET_COL].value_counts())

Shape: (45000, 14)
Columns: ['person_age', 'person_gender', 'person_education', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file', 'loan_status']
Missing values per column:
person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64

Target distribution:
loan_status
0    35000
1    10000
Name: count, dtype: int64


In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

train_raw = X_train.copy()
train_raw[TARGET_COL] = y_train
test_raw = X_test.copy()
test_raw[TARGET_COL] = y_test

train_raw.to_csv(DATA_DIR / "train_raw.csv", index=False)
test_raw.to_csv(DATA_DIR / "test_raw.csv", index=False)

# Optional top-level export that will be useful for later Streamlit step.
test_raw.to_csv(ROOT / "test_data.csv", index=False)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Saved raw split CSVs to data folder and test_data.csv at project root.")

Train shape: (36000, 13)
Test shape: (9000, 13)
Saved raw split CSVs to data folder and test_data.csv at project root.


In [12]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

train_processed = X_train_processed_df.copy()
train_processed[TARGET_COL] = y_train.values
test_processed = X_test_processed_df.copy()
test_processed[TARGET_COL] = y_test.values

train_processed.to_csv(DATA_DIR / "train_processed.csv", index=False)
test_processed.to_csv(DATA_DIR / "test_processed.csv", index=False)

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Processed train shape:", X_train_processed_df.shape)
print("Processed test shape:", X_test_processed_df.shape)

Numeric features: 8
Categorical features: 5
Processed train shape: (36000, 27)
Processed test shape: (9000, 27)


In [13]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", random_state=RANDOM_STATE),
}

results = []

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model),
        ]
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_score = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_score = pipeline.decision_function(X_test)

    metrics_row = {
        "ML Model Name": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_score),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }
    results.append(metrics_row)

    model_file = MODELS_DIR / f"{model_name.lower().replace(' ', '_')}.joblib"
    joblib.dump(pipeline, model_file)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1", ascending=False).reset_index(drop=True)

for metric_col in ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]:
    results_df[metric_col] = results_df[metric_col].round(4)

results_df.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)
results_df

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Support Vector Machine,0.9178,0.9642,0.8575,0.7555,0.8033,0.7540
1,Decision Tree,0.8977,0.8537,0.7672,0.7745,0.7708,0.7050
2,Logistic Regression,0.8993,0.9562,0.7888,0.7470,0.7673,0.7036
3,KNN,0.8956,0.9246,0.7909,0.7205,0.7541,0.6891
4,Naive Bayes,0.7356,0.9398,0.4566,0.9990,0.6267,0.5484


In [14]:
print("Saved model files:")
for p in sorted(MODELS_DIR.glob("*.joblib")):
    print("-", p.name)

print("\nSaved data splits:")
for p in sorted(DATA_DIR.glob("*.csv")):
    print("-", p.name)

print("\nMetrics file:", (RESULTS_DIR / "model_metrics.csv").name)

Saved model files:
- decision_tree.joblib
- knn.joblib
- logistic_regression.joblib
- naive_bayes.joblib
- preprocessor.joblib
- support_vector_machine.joblib

Saved data splits:
- loan_data.csv
- test_processed.csv
- test_raw.csv
- train_processed.csv
- train_raw.csv

Metrics file: model_metrics.csv
